# Python OS and Sys Module Exercises: 30 Coding Problems with Solutions

A practice notebook on the `os` and `sys` modules — files, directories, paths, environment variables, process info, command-line arguments, platform detection, and the import system — each with a concept note, a hint, a solution, and an explanation.

*Adapted for practice from the exercise list at [PYnative](https://pynative.com/python-os-sys-module-exercises/). Each code cell creates its own sample file(s)/folder(s) first, so the notebook runs standalone from top to bottom.*

---

## Concepts you'll need

This set covers two of Python's most-used standard library modules for scripting and automation.

**From `os`:**
- **Directories** — `os.getcwd()`, `os.listdir()`, `os.mkdir()` (one level, fails if parents missing) vs. `os.makedirs(path, exist_ok=True)` (creates the whole chain, no error if it already exists).
- **Files** — `os.rename()`, `os.remove()` (files only — use `os.rmdir()` for an empty directory, `shutil.rmtree()` for a non-empty one).
- **Paths** — `os.path.exists()`/`isfile()`/`isdir()` for checking, `os.path.join()` for building portable paths (never concatenate with `+`), `os.path.abspath()` to resolve a relative path, `os.path.splitext()` to split a filename into `(root, .ext)`, `os.path.dirname()`/`basename()` to split into folder/filename, `os.path.getsize()` for byte size.
- **Traversal** — `os.walk(top)` yields `(dirpath, dirnames, filenames)` for every directory in a tree, recursively — the standard tool for any "process every file under this folder" task.
- **Environment & process** — `os.environ` (dict-like; use `.get(key, default)` for safety), `os.getpid()`/`getppid()`, `os.system(command)` (runs a shell command, returns its exit code, but doesn't capture output).

**From `sys`:**
- **Interpreter info** — `sys.version` (full string) vs. `sys.version_info` (a named tuple — prefer this for version *comparisons* in code).
- **Command-line arguments** — `sys.argv` is a list where index 0 is always the script name and index 1+ are user-supplied arguments (all as strings).
- **Program control** — `sys.exit(message_or_code)` terminates the program; a string argument prints to stderr and exits with code 1, an int sets the exit code directly (0 = success).
- **Platform detection** — `sys.platform` (short: `"linux"`, `"darwin"`, `"win32"`) vs. the `platform` module's `platform.system()` (clean name) and `platform.platform()` (full descriptive string).
- **Import machinery** — `sys.path` is the list of directories Python searches for modules, in order; `sys.path.insert(0, dir)` adds a custom directory so it's checked first.
- **Recursion** — `sys.getrecursionlimit()`/`setrecursionlimit()` control how many nested function calls are allowed before Python raises `RecursionError`.

**A note on this notebook:** several exercises (20, 21, 30) are designed to read `sys.argv` from a real terminal invocation like `python script.py hello world`. Since a notebook has no command line of its own, those cells simulate the arguments by assigning directly to `sys.argv` first, with a comment explaining what the equivalent terminal command would look like.

Each exercise below gives a problem, a hint, a solution, and an explanation.

## Exercise 1. Print Current Directory

**Concept:** os.getcwd()

**Problem:** Print the current working directory.

**Given:**
```
no input — reads the environment directly
```

**Expected Output:**
```
Current Directory: /some/path (varies by system)
```

**Hint:** os.getcwd() returns an absolute path string with no arguments needed.

In [ ]:
import os

cwd = os.getcwd()
print("Current Directory:", cwd)

**Explanation:** os.getcwd() ("get current working directory") returns the absolute path of the directory the script is running from, as a plain string — no arguments needed. This is the directory relative paths are resolved against for the rest of the script's file operations.

## Exercise 2. List Directory Contents

**Concept:** os.listdir()

**Problem:** List all files and folders in a given directory.

**Given:**
```
path = "."
```

**Expected Output:**
```
A printed list of file and folder names (output varies by system)
```

**Hint:** listdir() returns both files and subdirectories, but does not recurse into subdirectories.

In [ ]:
import os

path = "."
contents = os.listdir(path)

for item in sorted(contents):
    print(item)

**Explanation:** os.listdir(path) returns a plain list of names — both files and subfolders — found directly inside path, without descending into any subdirectories (that's what os.walk(), in Exercise 16, is for). The entries aren't sorted by default, which is why sorted() is wrapped around the call here for consistent, readable output.

## Exercise 3. Create a Directory

**Concept:** os.path.exists() guard + os.mkdir()

**Problem:** Create a new folder only if it doesn't already exist.

**Given:**
```
folder_name = "test_folder"
```

**Expected Output:**
```
Folder 'test_folder' created successfully.
```

**Hint:** Checking os.path.exists() first avoids a FileExistsError from mkdir().

In [ ]:
import os

folder_name = "test_folder"

if not os.path.exists(folder_name):
    os.mkdir(folder_name)
    print(f"Folder '{folder_name}' created successfully.")
else:
    print(f"Folder '{folder_name}' already exists.")

**Explanation:** os.path.exists() returns True for either a file or a folder at that path, so checking it before creation prevents a FileExistsError. os.mkdir() creates exactly one new directory level — it raises FileNotFoundError if any parent directory in the path doesn't already exist, which is why Exercise 4 introduces os.makedirs() for nested paths.

## Exercise 4. Create Nested Directories

**Concept:** os.makedirs(path, exist_ok=True)

**Problem:** Create a nested directory structure a/b/c in one call, even if none of the parents exist yet.

**Given:**
```
nested_path = "a/b/c"
```

**Expected Output:**
```
Nested directories 'a/b/c' created successfully.
```

**Hint:** makedirs() creates every intermediate level; exist_ok=True suppresses the error if the path is already there.

In [ ]:
import os

nested_path = os.path.join("a", "b", "c")

os.makedirs(nested_path, exist_ok=True)
print(f"Nested directories '{nested_path}' created successfully.")

**Explanation:** os.path.join("a", "b", "c") builds the path using the correct separator for the current OS, making the script portable. Unlike os.mkdir(), which fails if a parent directory is missing, os.makedirs() creates the entire chain in one call; exist_ok=True additionally suppresses the FileExistsError that would otherwise fire if any part of the path already exists.

## Exercise 5. Rename a File

**Concept:** os.path.exists() guard + os.rename()

**Problem:** Rename a file, checking first that the source exists.

**Given:**
```
old.txt exists in the current directory
```

**Expected Output:**
```
Renamed 'old.txt' to 'new.txt' successfully.
```

**Hint:** os.rename(src, dst) renames or moves a file, and is atomic on most systems.

In [ ]:
import os

src = "old.txt"
dst = "new.txt"

open(src, "w").close()

if os.path.exists(src):
    os.rename(src, dst)
    print(f"Renamed '{src}' to '{dst}' successfully.")
else:
    print(f"Source file '{src}' does not exist.")

**Explanation:** open(src, "w").close() creates an empty zero-byte file purely for demonstration purposes. os.path.exists(src) guards against a FileNotFoundError before attempting the rename. os.rename(src, dst) is atomic on most operating systems — if dst already existed, it would be silently overwritten on Unix but raise an error on Windows, a platform difference worth knowing about.

## Exercise 6. Delete a File

**Concept:** os.path.exists() guard + os.remove()

**Problem:** Delete a file, but only after confirming it exists.

**Given:**
```
temp.txt (created in the script)
```

**Expected Output:**
```
File 'temp.txt' deleted successfully.
```

**Hint:** os.remove() works only on files — IsADirectoryError is raised if you point it at a folder.

In [ ]:
import os

filename = "temp.txt"

with open(filename, "w") as f:
    f.write("temporary content")

if os.path.exists(filename):
    os.remove(filename)
    print(f"File '{filename}' deleted successfully.")
else:
    print(f"File '{filename}' not found.")

**Explanation:** os.path.exists() branches safely before the destructive operation. os.remove() permanently deletes the specified file — it raises IsADirectoryError if pointed at a folder instead, since deleting directories requires a different function (os.rmdir() for empty ones, shutil.rmtree() for non-empty ones, as shown in the next exercise).

## Exercise 7. Delete a Directory Tree

**Concept:** shutil.rmtree() for recursive deletion

**Problem:** Create a nested directory with a file inside, then remove the entire tree at once.

**Given:**
```
cleanup/a/b/note.txt (created in the script)
```

**Expected Output:**
```
Directory tree 'cleanup' removed successfully.
```

**Hint:** os.rmdir() only removes EMPTY directories; shutil.rmtree() removes everything inside recursively.

In [ ]:
import os
import shutil

base_dir = "cleanup"
nested_path = os.path.join(base_dir, "a", "b")

os.makedirs(nested_path, exist_ok=True)
with open(os.path.join(nested_path, "note.txt"), "w") as f:
    f.write("temporary note")

if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
    print(f"Directory tree '{base_dir}' removed successfully.")
else:
    print(f"Directory '{base_dir}' not found.")

**Explanation:** os.makedirs(nested_path, exist_ok=True) builds the full cleanup/a/b path in one call. shutil.rmtree(base_dir) recursively deletes the entire tree — every file and subdirectory inside it — which os.rmdir() could not do here since it only succeeds on already-empty directories. shutil ("shell utilities") complements os with these higher-level, recursive file operations.

## Exercise 8. Check Path Existence

**Concept:** os.path.exists() / isfile() / isdir()

**Problem:** Check whether a path exists and print a descriptive message.

**Given:**
```
path = "sample.txt" (created in the script)
```

**Expected Output:**
```
Path 'sample.txt' exists.
```

**Hint:** isfile() and isdir() give more specific checks than the general-purpose exists().

In [ ]:
import os

path = "sample.txt"

with open(path, "w") as f:
    f.write("hello")

if os.path.exists(path):
    print(f"Path '{path}' exists.")
else:
    print(f"Path '{path}' does not exist.")

print("Is a file?", os.path.isfile(path))
print("Is a directory?", os.path.isdir(path))

**Explanation:** os.path.exists() answers 'is there anything at all here' for either a file or a directory (and returns False for a broken symlink). os.path.isfile() and os.path.isdir() narrow that down — isfile() is True only for regular files, isdir() only for directories — letting you build precise validation logic when it matters which kind of thing you're dealing with.

## Exercise 9. Split File Extension

**Concept:** os.path.splitext()

**Problem:** Split a filename into its base name and extension.

**Given:**
```
filename = "report_2025.pdf"
```

**Expected Output:**
```
Base: report_2025
Extension: .pdf
```

**Hint:** splitext() returns a 2-tuple; the extension includes the leading dot.

In [ ]:
import os

filename = "report_2025.pdf"

base, ext = os.path.splitext(filename)

print("Base:", base)
print("Extension:", ext)

print("\nEdge cases:")
print("splitext('README')     :", os.path.splitext("README"))
print("splitext('.gitignore') :", os.path.splitext(".gitignore"))

**Explanation:** os.path.splitext() splits at the LAST dot in the filename and returns a 2-tuple (root, ext), with the extension keeping its leading dot — ('report_2025', '.pdf') here. A file with no extension at all (like 'README') gets an empty string for ext, and a dotfile like '.gitignore' is treated as having no extension either, since the leading dot is part of the name, not a separator.

## Exercise 10. Get File Size

**Concept:** os.path.getsize()

**Problem:** Create a file and check its size in bytes.

**Given:**
```
data.txt containing "Hello, Python!" (created in the script)
```

**Expected Output:**
```
File size: 14 bytes
```

**Hint:** getsize() reads filesystem metadata directly — it never needs to open or read the file's content.

In [ ]:
import os

filepath = "data.txt"

with open(filepath, "w") as f:
    f.write("Hello, Python!")

size = os.path.getsize(filepath)
print(f"File size: {size} bytes")

**Explanation:** os.path.getsize() returns a file's size in bytes purely from filesystem metadata, with no need to open or read its actual content. "Hello, Python!" has 14 characters, and since each ASCII character takes exactly 1 byte in UTF-8, the byte count and character count match here — for text containing non-ASCII Unicode characters, the byte size can be larger than the character count.

## Exercise 11. Read an Environment Variable

**Concept:** os.environ.get(key, default)

**Problem:** Read the PATH environment variable safely, with a fallback if it's missing.

**Given:**
```
no input — reads the OS environment
```

**Expected Output:**
```
The value of PATH (a long, separator-joined string)
```

**Hint:** .get() returns a fallback default instead of raising KeyError, unlike direct bracket access.

In [ ]:
import os

path_value = os.environ.get("PATH", "PATH variable not set")
print("PATH:", path_value[:200], "...")

directories = path_value.split(os.pathsep)
print(f"\nNumber of directories in PATH: {len(directories)}")
for d in directories[:5]:
    print(" ", d)

**Explanation:** os.environ is a dict-like object mapping variable names to their string values, snapshotted from when the interpreter started. .get(key, default) is the safe way to read it — if PATH weren't set, the fallback string would be returned instead of raising a KeyError, which direct bracket access (os.environ["PATH"]) would do. os.pathsep gives the platform-correct separator (: on Unix, ; on Windows) for splitting PATH into its individual directories.

## Exercise 12. Set an Environment Variable

**Concept:** os.environ[key] = value

**Problem:** Set a custom environment variable, then read it back.

**Given:**
```
APP_MODE = "development"
```

**Expected Output:**
```
APP_MODE is set to: development
```

**Hint:** Both the key and value assigned to os.environ must be strings, or Python raises a TypeError.

In [ ]:
import os

os.environ["APP_MODE"] = "development"

app_mode = os.environ.get("APP_MODE", "not set")
print(f"APP_MODE is set to: {app_mode}")

**Explanation:** os.environ["APP_MODE"] = "development" adds or updates that key in the current process's environment — both key and value must be strings, so assigning an int directly would raise a TypeError. This change only affects the current Python process (and any child processes it spawns via subprocess); it never persists back to the parent shell once the script exits.

## Exercise 13. List All Environment Variables

**Concept:** os.environ.items(), sorted for readability

**Problem:** Print every currently available environment variable.

**Given:**
```
no input — reads the live process environment
```

**Expected Output:**
```
A list of NAME = value pairs, one per line (varies by system)
```

**Hint:** os.environ behaves exactly like a dict, so .items() works the same way it does on any dictionary.

In [ ]:
import os

count = 0
for key, value in sorted(os.environ.items()):
    print(f"{key} = {value[:60]}{'...' if len(value) > 60 else ''}")
    count += 1
    if count >= 10:
        print(f"... and {len(os.environ) - 10} more")
        break

**Explanation:** os.environ.items() yields (name, value) tuples exactly like calling .items() on a regular dict. sorted() arranges them alphabetically by key for easier scanning, rather than the arbitrary insertion order the raw environment would otherwise give. Worth remembering: the environment may hold sensitive values like API keys, so logging or displaying it fully is best avoided in shared or production contexts (this cell truncates values and caps the count for that reason).

## Exercise 14. Get Current Process ID

**Concept:** os.getpid() and os.getppid()

**Problem:** Print the current process's ID and its parent process's ID.

**Given:**
```
no input — assigned by the OS
```

**Expected Output:**
```
Current Process ID: 12345 (varies each run)
```

**Hint:** Each run of the script gets a different PID, assigned fresh by the operating system.

In [ ]:
import os

pid = os.getpid()
ppid = os.getppid()

print(f"Current Process ID: {pid}")
print(f"Parent Process ID: {ppid}")

**Explanation:** os.getpid() returns the integer process ID the OS assigned to the current Python interpreter instance — a fresh, different number each time the script runs. os.getppid() returns the PID of whatever launched this process (typically a shell or, in this notebook's case, the kernel process). PIDs are commonly embedded in log lines (like f"[PID {os.getpid()}] ...") to distinguish output when the same script runs concurrently in multiple processes.

## Exercise 15. Run a Shell Command

**Concept:** os.system() and its integer exit code

**Problem:** Run a shell command and check its exit code.

**Given:**
```
a directory-listing command appropriate to the OS
```

**Expected Output:**
```
Command output printed directly, followed by Exit code: 0
```

**Hint:** 0 conventionally means success; any non-zero value signals the command failed.

In [ ]:
import os
import sys

if sys.platform == "win32":
    command = "dir"
else:
    command = "ls"

exit_code = os.system(command)
print(f"\nExit code: {exit_code}")

**Explanation:** os.system(command) hands the string straight to the OS's default shell and runs it, printing its output directly to the terminal rather than capturing it as a Python value. The integer it returns is the command's exit status — 0 conventionally means success, any non-zero value signals some kind of failure. Because os.system() can't capture output into a variable, the subprocess module (specifically subprocess.run(command, capture_output=True, text=True)) is the modern, more flexible choice when you need the actual output, not just the exit code.

## Exercise 16. Walk a Directory Tree

**Concept:** os.walk() for automatic recursive traversal

**Problem:** Recursively list every file in a directory tree.

**Given:**
```
a sample tree with two subdirectories, each containing one file, plus one file at the root
```

**Expected Output:**
```
Full path of every file found, one per line
```

**Hint:** os.walk() yields a (dirpath, dirnames, filenames) tuple for every directory it visits, recursively.

In [ ]:
import os

os.makedirs("walk_demo/subdir_a", exist_ok=True)
os.makedirs("walk_demo/subdir_b", exist_ok=True)

with open("walk_demo/subdir_a/file1.txt", "w") as f:
    f.write("file one")
with open("walk_demo/subdir_b/file2.txt", "w") as f:
    f.write("file two")
with open("walk_demo/root_file.txt", "w") as f:
    f.write("root file")

for dirpath, dirnames, filenames in os.walk("walk_demo"):
    for filename in filenames:
        full_path = os.path.join(dirpath, filename)
        print(full_path)

**Explanation:** os.walk("walk_demo") is a generator that handles all the recursion itself, yielding one tuple per directory visited: the current path, the subdirectory names within it, and the filenames within it. os.path.join(dirpath, filename) combines those into a complete, usable path — concatenating with plain string + would break on Windows, where the path separator is \ rather than /. dirnames can even be modified in-place during iteration to prune branches you want os.walk() to skip entirely.

## Exercise 17. Join Paths Safely

**Concept:** os.path.join() — never build paths with string concatenation

**Problem:** Build a file path from separate components in a portable way.

**Given:**
```
base = "projects", subfolder = "python_exercises", filename = "solution.py"
```

**Expected Output:**
```
projects/python_exercises/solution.py (Unix) or projects\\python_exercises\\solution.py (Windows)
```

**Hint:** If any argument to join() is an absolute path, everything before it in the call gets discarded.

In [ ]:
import os

base = "projects"
subfolder = "python_exercises"
filename = "solution.py"

full_path = os.path.join(base, subfolder, filename)
print("Joined path:", full_path)

abs_path = os.path.abspath(full_path)
print("Absolute path:", abs_path)

print("\nAbsolute-path override rule:")
print(os.path.join("home", "/etc", "config"))

**Explanation:** os.path.join() concatenates its arguments using the correct separator for whatever OS the code is actually running on, which is what makes it portable in a way plain string concatenation never could be. A subtle gotcha worth knowing: if any argument is itself an absolute path (starting with / on Unix, or a drive letter on Windows), every component before it is silently discarded — os.path.join("home", "/etc", "config") returns "/etc/config", not "home/etc/config".

## Exercise 18. Get Absolute Path

**Concept:** os.path.abspath(), dirname(), basename()

**Problem:** Convert a relative path to absolute, then split it into directory and filename parts.

**Given:**
```
relative_path = "data/report.csv"
```

**Expected Output:**
```
An absolute path, plus its directory portion and filename portion printed separately
```

**Hint:** abspath() also normalizes any . or .. components in the path along the way.

In [ ]:
import os

relative_path = "data/report.csv"

abs_path = os.path.abspath(relative_path)
directory = os.path.dirname(abs_path)
filename = os.path.basename(abs_path)

print("Absolute path:", abs_path)
print("Directory    :", directory)
print("Filename     :", filename)

**Explanation:** os.path.abspath() prepends the current working directory to a relative path, and also normalizes any . or .. segments along the way, so "data/../data/report.csv" would collapse into a clean path. os.path.dirname() returns everything except the final path component (the containing folder), and os.path.basename() returns only that final component (the filename) — together they let you split any path into its folder and file parts without manual string slicing.

## Exercise 19. Print Python Version

**Concept:** sys.version (string) vs. sys.version_info (named tuple)

**Problem:** Display the Python version both as a readable string and as individual numeric components.

**Given:**
```
no input — read from the running interpreter
```

**Expected Output:**
```
Python version: 3.x.x ...
Major: 3  Minor: x  Micro: x
```

**Hint:** Always use version_info (not string-parsing of sys.version) for version-comparison logic in real code.

In [ ]:
import sys

print("Python version:", sys.version)
print(f"Major: {sys.version_info.major}  "
      f"Minor: {sys.version_info.minor}  "
      f"Micro: {sys.version_info.micro}")

if sys.version_info.major < 3:
    print("Warning: Python 3 required")
else:
    print("Running on a supported Python 3.x version.")

**Explanation:** sys.version is a plain string with the full version plus build metadata like compile date and compiler name — readable for logging, but awkward to parse programmatically. sys.version_info is a named tuple with major, minor, micro, releaselevel, and serial fields, giving clean, direct access for version-comparison logic like `if sys.version_info.major < 3:` — always prefer this over parsing the sys.version string, since its exact format can vary slightly across platforms and builds.

## Exercise 20. Read Command-Line Arguments

**Concept:** sys.argv — index 0 is the script name, arguments start at index 1

**Problem:** Read and print all arguments passed on the command line, with their index position.

**Given:**
```
terminal invocation: python solution.py hello world 42
```

**Expected Output:**
```
[0] solution.py
[1] hello
[2] world
[3] 42
```

**Hint:** All values in sys.argv are strings — convert explicitly (e.g. int(sys.argv[3])) if you need a number.

In [ ]:
import sys

# In a real terminal you'd run: python solution.py hello world 42
# Since a notebook has no command line of its own, we simulate that call here:
sys.argv = ["solution.py", "hello", "world", "42"]

print(f"Total arguments: {len(sys.argv)}")
print("Arguments received:")

for index, arg in enumerate(sys.argv):
    print(f"  [{index}] {arg}")

**Explanation:** sys.argv is a list the interpreter populates before the script runs — sys.argv[0] is always the script's own name/path, so any arguments the user actually typed start at index 1. Checking len(sys.argv) before accessing a specific index (as Exercise 21 does) avoids an IndexError when fewer arguments were provided than expected. enumerate(sys.argv) pairs each argument with its position automatically, without a manual counter.

## Exercise 21. Exit a Program

**Concept:** sys.exit() for controlled, signaling termination

**Problem:** Require a command-line argument and exit gracefully with an error message if it's missing.

**Given:**
```
terminal invocation with an argument: python solution.py Alice
```

**Expected Output:**
```
Hello, Alice!
```

**Hint:** sys.exit("message") prints to stderr and exits with code 1; sys.exit(0) signals success explicitly.

In [ ]:
import sys

def run(argv):
    if len(argv) < 2:
        sys.exit("Error: Please provide a name as an argument.")
    name = argv[1]
    print(f"Hello, {name}!")

# Simulates: python solution.py Alice
run(["solution.py", "Alice"])

# Simulates: python solution.py   (missing the required argument)
try:
    run(["solution.py"])
except SystemExit as e:
    print(f"Program would exit here with: {e}")

**Explanation:** len(sys.argv) < 2 catches the case where no user-supplied argument was given, since sys.argv[0] (the script name) is always present — this check must come before touching sys.argv[1] to avoid an IndexError. sys.exit("message"), when given a string, prints that message to stderr and terminates the process with exit code 1, which a calling shell or CI pipeline reads as failure. (This cell wraps the second call in try/except SystemExit purely so the notebook can show what would happen without actually halting the kernel — in a real script, sys.exit() would simply end the program there.)

## Exercise 22. Get Platform Info

**Concept:** sys.platform vs. the platform module

**Problem:** Detect and describe the current operating system in several levels of detail.

**Given:**
```
no input — detected from the runtime environment
```

**Expected Output:**
```
Platform: linux (or "darwin"/"win32"), plus a detailed description
```

**Hint:** sys.platform gives a short code for branching logic; the platform module gives human-readable detail.

In [ ]:
import sys
import platform

print("sys.platform       :", sys.platform)
print("platform.system()  :", platform.system())
print("platform.platform():", platform.platform())
print("platform.machine() :", platform.machine())

**Explanation:** sys.platform is a short, lowercase string set at interpreter startup — "linux", "darwin" (macOS), or "win32" (all Windows versions, including 64-bit) — and is the simplest way to branch platform-specific logic. platform.system() gives a cleaner, title-cased name ("Linux", "Darwin", "Windows") more suited to user-facing output. platform.platform() returns a full descriptive string useful in diagnostic logs, and platform.machine() gives the hardware architecture (like "x86_64" or "arm64"), handy when building or distributing architecture-specific packages.

## Exercise 23. Inspect sys.path

**Concept:** sys.path — the ordered list Python searches for modules

**Problem:** Print every directory Python searches when resolving an import statement.

**Given:**
```
no input — maintained by the interpreter
```

**Expected Output:**
```
A numbered list of search-path directories (varies by system/virtual environment)
```

**Hint:** An empty string entry in sys.path represents the current working directory.

In [ ]:
import sys

print(f"Total search paths: {len(sys.path)}\n")

for index, path in enumerate(sys.path):
    print(f"[{index}] {path if path else '(current directory)'}")

**Explanation:** sys.path is the ordered list of directories Python scans, in sequence, whenever an import statement runs — the first matching module found wins, so if two directories both contain a same-named module, the earlier (lower-indexed) one silently takes precedence, a common source of shadowing bugs. It's assembled from the script's own directory, the PYTHONPATH environment variable, and installation defaults. An empty string "" appearing anywhere in the list specifically represents the current working directory, which the conditional expression here makes explicit rather than printing a confusing blank line.

## Exercise 24. Add to sys.path

**Concept:** sys.path.insert(0, dir) to import from a non-standard location

**Problem:** Add a custom directory to sys.path so a module inside it can be imported directly.

**Given:**
```
a custom_modules/greet.py file created at runtime
```

**Expected Output:**
```
Hello from custom module!
```

**Hint:** insert(0, ...) puts the directory first in the search order, checked before any other location.

In [ ]:
import sys
import os

custom_dir = os.path.abspath("custom_modules")
os.makedirs(custom_dir, exist_ok=True)

with open(os.path.join(custom_dir, "greet.py"), "w") as f:
    f.write('def hello():\n    print("Hello from custom module!")\n')

if custom_dir not in sys.path:
    sys.path.insert(0, custom_dir)

import greet
greet.hello()

**Explanation:** sys.path.insert(0, custom_dir) places the custom directory at the very front of the search list, so it's the first location checked for any subsequent import — using .append() instead would add it to the end, meaning it would only be consulted after every standard-library and site-packages location had already been searched. os.path.abspath() converts the directory name to a full path so the sys.path entry stays valid even if the working directory changes later. The `if custom_dir not in sys.path` guard prevents duplicate entries from piling up if this code runs more than once.

## Exercise 25. Get Recursion Limit

**Concept:** sys.getrecursionlimit()

**Problem:** Retrieve Python's maximum allowed recursion depth.

**Given:**
```
no input
```

**Expected Output:**
```
Current recursion limit: 1000 (the default on most installations)
```

**Hint:** Every nested function call — direct or indirect — counts as one level against this limit.

In [ ]:
import sys

limit = sys.getrecursionlimit()
print(f"Current recursion limit: {limit}")

def countdown(n):
    if n == 0:
        return "Done"
    return countdown(n - 1)

print(countdown(50))
print("Recursion test with depth 50: passed")

**Explanation:** sys.getrecursionlimit() returns the maximum depth the Python call stack is allowed to reach — 1000 by default on most installations, meaning a chain of more than 1000 nested calls raises a RecursionError. This isn't just direct self-calls: every function call, including calls made indirectly through other functions, counts as one level toward the limit. Real-world recursive tasks like deep tree traversal or parsing deeply nested JSON can genuinely exceed 1000 levels on large enough inputs, which is exactly the scenario Exercise 26 addresses.

## Exercise 26. Change Recursion Limit

**Concept:** sys.setrecursionlimit()

**Problem:** Raise the recursion limit and verify a deeper recursive call now succeeds.

**Given:**
```
a recursive call with depth 1500 (which would fail under the default 1000 limit)
```

**Expected Output:**
```
Old limit: 1000, New limit: 2000, Recursion test with depth 1500: passed
```

**Hint:** Setting the limit far too high risks a hard interpreter crash rather than a clean, catchable RecursionError.

In [ ]:
import sys

old_limit = sys.getrecursionlimit()
print(f"Old limit: {old_limit}")

sys.setrecursionlimit(2000)
print(f"New limit: {sys.getrecursionlimit()}")

def deep_recurse(n):
    if n == 0:
        return "Done"
    return deep_recurse(n - 1)

result = deep_recurse(1500)
print(f"Recursion test with depth 1500: passed")

sys.setrecursionlimit(old_limit)
print(f"Limit restored to: {sys.getrecursionlimit()}")

**Explanation:** sys.setrecursionlimit(2000) raises the ceiling immediately, letting the subsequent depth-1500 call complete without raising RecursionError, where it would have failed under the original default of 1000. Saving old_limit beforehand and restoring it afterward is good practice, especially inside libraries or test suites, since permanently altering global interpreter state can have unexpected side effects elsewhere. Setting the limit extremely high (like 10**6) is genuinely risky — it can cause a hard crash (segmentation fault) once the OS-level thread stack itself is exhausted, before Python ever gets a chance to raise its own clean, catchable error.

## Exercise 27. Script Info Logger

**Concept:** combining sys and os to log runtime context; sys.stdout.write()

**Problem:** Collect script name, working directory, and platform, and write them to a log file.

**Given:**
```
no external input — all values derived from the runtime environment
```

**Expected Output:**
```
script_info.log created, containing Script/Directory/Platform/Python lines
```

**Hint:** sys.stdout.write() doesn't add a newline automatically, unlike print() — full manual control over line endings.

In [ ]:
import os
import sys

script_name = sys.argv[0] if sys.argv else "(interactive session)"
current_dir = os.getcwd()
platform_id = sys.platform
python_ver  = sys.version.split()[0]

log_lines = [
    f"Script   : {script_name}",
    f"Directory: {current_dir}",
    f"Platform : {platform_id}",
    f"Python   : {python_ver}",
]

log_path = "script_info.log"

with open(log_path, "w") as log_file:
    for line in log_lines:
        log_file.write(line + "\n")
        sys.stdout.write(line + "\n")

print(f"\nLog written to: {os.path.abspath(log_path)}")

**Explanation:** sys.argv[0] holds the running script's own name or path — in an interactive session (like this notebook's kernel) it may be empty or a special placeholder rather than a real filename, which is why a fallback is included here. sys.version.split()[0] extracts just the version number, discarding the trailing build-date and compiler details. sys.stdout.write(line + "\n") writes directly to the standard output stream without print()'s automatic newline and space-joining behavior, making explicit that both the log file and the terminal are ultimately just two different file-like streams being written to.

## Exercise 28. Directory Size Calculator

**Concept:** os.walk() + os.path.getsize() combined to total a tree's size

**Problem:** Calculate the total size of every file in a directory tree.

**Given:**
```
a sample tree with files of known sizes (1KB, 2KB, 0.5KB)
```

**Expected Output:**
```
Total size: 3584 bytes / 3.50 KB / 0.0034 MB
```

**Hint:** getsize() only reads filesystem metadata per file — no file content is ever opened or read.

In [ ]:
import os

os.makedirs("size_demo/docs", exist_ok=True)
os.makedirs("size_demo/data", exist_ok=True)

with open("size_demo/docs/readme.txt", "w") as f:
    f.write("A" * 1024)

with open("size_demo/data/records.csv", "w") as f:
    f.write("B" * 2048)

with open("size_demo/notes.txt", "w") as f:
    f.write("C" * 512)

total_bytes = 0
for dirpath, dirnames, filenames in os.walk("size_demo"):
    for filename in filenames:
        full_path = os.path.join(dirpath, filename)
        total_bytes += os.path.getsize(full_path)

print(f"Total size: {total_bytes} bytes")
print(f"Total size: {total_bytes / 1024:.2f} KB")
print(f"Total size: {total_bytes / 1024 ** 2:.4f} MB")

**Explanation:** os.walk() supplies every file in the tree; os.path.getsize() adds each one's byte size to a running total, reading only filesystem metadata rather than opening or reading any actual file content. os.path.join(dirpath, filename) builds the correct full path for each file — passing just the bare filename to getsize() would fail unless that file happened to sit in the current working directory. Dividing by 1024 converts to kibibytes and by 1024**2 to mebibytes, with :.2f/:.4f rounding the float display to a clean, readable precision.

## Exercise 29. Environment Config Loader

**Concept:** a realistic os.environ.get() + defaults pattern for app configuration

**Problem:** Load application settings from environment variables with sensible fallback defaults.

**Given:**
```
APP_HOST and APP_PORT set; APP_DEBUG and APP_ENV intentionally left unset
```

**Expected Output:**
```
A formatted configuration summary, with APP_DEBUG/APP_ENV falling back to their defaults
```

**Hint:** Separating the loading step (building the config dict) from the display step is a common, cleaner pattern.

In [ ]:
import os
import sys

os.environ["APP_HOST"] = "127.0.0.1"
os.environ["APP_PORT"] = "8080"
# APP_DEBUG is intentionally left unset to demonstrate the fallback

config = {
    "APP_HOST":  os.environ.get("APP_HOST",  "localhost"),
    "APP_PORT":  os.environ.get("APP_PORT",  "5000"),
    "APP_DEBUG": os.environ.get("APP_DEBUG", "False"),
    "APP_ENV":   os.environ.get("APP_ENV",   "production"),
}

sys.stdout.write("=== Application Configuration ===\n")
for key, value in config.items():
    sys.stdout.write(f"  {key:<12}: {value}\n")
sys.stdout.write("=================================\n")

**Explanation:** os.environ.get(key, default) guarantees every setting has a usable value regardless of what's actually configured in the deployment environment — APP_HOST and APP_PORT pick up the values explicitly set above, while APP_DEBUG and APP_ENV fall back to their defaults since they were never set. Collecting everything into one config dictionary first, and only printing it afterward, cleanly separates the loading logic from the display logic — the same underlying pattern real configuration libraries like python-decouple use. The f"{key:<12}" format left-aligns each key within a 12-character field, producing a neatly aligned two-column layout.

## Exercise 30. Recursive File Finder

**Concept:** a capstone combining sys.argv, os.walk(), and os.path.splitext()

**Problem:** Search a directory tree recursively for every file matching a given extension.

**Given:**
```
terminal invocation: python solution.py .txt
```

**Expected Output:**
```
Full paths of every matching file, followed by a total count
```

**Hint:** Normalize the extension (lowercase, ensure it starts with a dot) so both 'txt' and '.TXT' work as input.

In [ ]:
import os
import sys

os.makedirs("search_demo/docs", exist_ok=True)
os.makedirs("search_demo/data", exist_ok=True)
for name in ["search_demo/docs/notes.txt",
             "search_demo/docs/readme.txt",
             "search_demo/data/records.csv",
             "search_demo/data/summary.txt",
             "search_demo/config.json"]:
    with open(name, "w") as f:
        f.write("sample content")

# Simulates: python solution.py .txt
sys.argv = ["solution.py", ".txt"]

if len(sys.argv) < 2:
    sys.exit("Usage: python solution.py <extension>  e.g. python solution.py .txt")

ext = sys.argv[1].lower()
if not ext.startswith("."):
    ext = "." + ext

matches = []
for dirpath, dirnames, filenames in os.walk("search_demo"):
    for filename in filenames:
        if os.path.splitext(filename)[1].lower() == ext:
            matches.append(os.path.join(dirpath, filename))

if matches:
    print(f"Files with extension '{ext}':")
    for path in matches:
        print(f"  {path}")
    print(f"\nTotal found: {len(matches)}")
else:
    print(f"No files with extension '{ext}' found.")

**Explanation:** This capstone combines several techniques from across the set: sys.argv for input, sys.exit() for validating that input, os.walk() for recursive traversal, os.path.join() for correct path construction, and os.path.splitext() for extension comparison. Normalizing ext with .lower() and ensuring a leading dot means the tool accepts 'txt', '.txt', or '.TXT' interchangeably and still matches correctly. Collecting matches into a list before printing (rather than printing inline during the walk) cleanly separates the search logic from the display logic, making it easy to later add sorting, size-filtering, or handing the list off to another function entirely.